Matrix Factorization using svd

In [1]:
import pandas as pd
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
from surprise import accuracy

In [2]:
ratings = pd.read_csv("../data/ratings.csv")

In [3]:
reader = Reader(rating_scale=(0.5, 5.0))

In [4]:
data = Dataset.load_from_df(
    ratings[["userId", "movieId", "rating"]],
    reader
)

In [5]:
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

In [6]:
model = SVD(
    n_factors=50,
    n_epochs=20,
    lr_all=0.005,
    reg_all=0.02
)


In [7]:
model.fit(trainset)

Evaluate model (RMSE)

In [15]:
predictions = model.test(testset)
rmse_svd=accuracy.rmse(predictions)

RMSE: 0.8768


Predict ratings for unseen movies

In [16]:
def get_unseen_movies(user_id, ratings_df):
    seen_movies = ratings_df[ratings_df["userId"] == user_id]["movieId"].unique()
    all_movies = ratings_df["movieId"].unique()
    return [m for m in all_movies if m not in seen_movies]


In [17]:
def recommend_movies_svd(user_id, n_recommendations=5):
    unseen_movies = get_unseen_movies(user_id, ratings)

    predictions = [
        (movie_id, model.predict(user_id, movie_id).est)
        for movie_id in unseen_movies
    ]

    predictions.sort(key=lambda x: x[1], reverse=True)
    return predictions[:n_recommendations]
recommend_movies_svd(user_id=1, n_recommendations=5)


[(318, 5.0), (58559, 5.0), (910, 5.0), (750, 5.0), (1201, 5.0)]

In [18]:
movies = pd.read_csv("../data/movies.csv")

def show_titles(recommendations):
    movie_ids = [rec[0] for rec in recommendations]
    return movies[movies["movieId"].isin(movie_ids)][["movieId", "title"]]


In [19]:
show_titles(recommend_movies_svd(1))


,movieId,title
277,318,"Shawshank Redemption, The (1994)"
602,750,Dr. Strangelove or: How I Learned to Stop Worr...
692,910,Some Like It Hot (1959)
903,1201,"Good, the Bad and the Ugly, The (Buono, il bru..."
6710,58559,"Dark Knight, The (2008)"


In [ ]:
from 
comparison = pd.DataFrame({
    "Method": ["User-based CF", "Item-based CF", "SVD"],
    "RMSE": [rmse_user, rmse_item, rmse_svd],
    "Pros": [
        "Simple and intuitive",
        "Stable and scalable",
        "Handles sparsity and learns hidden features"
    ],
    "Cons": [
        "Sparse data problem",
        "Cold start for new items",
        "Computationally complex"
    ]
})

comparison


In [24]:
import json

with open("../outputs/rmse_results.json") as f:
    results = json.load(f)

rmse_user = results["rmse_user"]
rmse_item = results["rmse_item"]

In [25]:
comparison = pd.DataFrame({
    "Method": ["User-based CF", "Item-based CF", "SVD"],
    "RMSE": [rmse_user, rmse_item, rmse_svd],
    "Pros": [
        "Simple and intuitive",
        "Stable and scalable",
        "Handles sparsity and learns hidden features"
    ],
    "Cons": [
        "Sparse data problem",
        "Cold start for new items",
        "Computationally complex"
    ]
})

comparison


,Method,RMSE,Pros,Cons
0,User-based CF,1.096305,Simple and intuitive,Sparse data problem
1,Item-based CF,0.905096,Stable and scalable,Cold start for new items
2,SVD,0.876816,Handles sparsity and learns hidden features,Computationally complex
